# Dataset 2 — Embedding Model Selection (p = 0)

Same selection logic as Dataset 1's `05_embedding_selection_p0_*`, applied to the **static** Dataset 2.
Compares all embedding candidates — GraphSAGE (feature_based) **v1 + v2** and Node2Vec (network_based) **v1 + v2**,
each at 32 / 64 / 128 dims — and picks the best embedding + model. `ModelTrainer` uses `split='random'`
(same as the combined threshold notebooks). Best model saved to `models/dataset_2/05_a`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    CLASSICAL_FEATURE_CANDIDATES,
    load_gnn_dataset,
    load_model,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
TARGET_COL = 'log_systemic_risk_label'
print('Project root:', PROJECT_ROOT)

## Load embedding candidates (v1 + v2, GraphSAGE + Node2Vec, 32/64/128)

In [ ]:
DIMS = [32, 64, 128]
EMB_FILES = {}
for v in ['v1', 'v2']:
    for d in DIMS:
        EMB_FILES[f'graphsage_{v}_{d}'] = f'graphsage_{v}_{d}_dataset2_dataset.parquet'
        EMB_FILES[f'node2vec_{v}_{d}']  = f'node2vec_{v}_{d}_dataset2_dataset.parquet'

trainers = {}
for key, fname in EMB_FILES.items():
    edf, ecols = load_gnn_dataset(PROJECT_ROOT, target_col=TARGET_COL, filename=fname)
    trainers[key] = ModelTrainer(df=edf, feature_cols=ecols, target_col=TARGET_COL, split='random')

pd.DataFrame({k: {'n_emb': len(t.feature_cols), 'train': len(t.train_df), 'val': len(t.val_df), 'test': len(t.test_df)}
             for k, t in trainers.items()}).T

## Define Models

In [ ]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(32,63,16), max_iter=300, activation="relu", learning_rate="adaptive", learning_rate_init=0.001, early_stopping=True, n_iter_no_change=3, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}
list(candidate_models)

## Train all candidates

In [ ]:
for key, t in trainers.items():
    t.train_all(candidate_models)
    print(f'\n=== {key} ===')
    display(t.leaderboard()[DISPLAY_COLS])

## Hyperparameter Tuning

In [ ]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([np.full(len(trainer.train_df), -1), np.zeros(len(trainer.val_df), dtype=int)])
    search = RandomizedSearchCV(base_model, param_distributions, n_iter=n_iter,
                                cv=PredefinedSplit(split_idx), scoring='neg_root_mean_squared_error',
                                random_state=42, n_jobs=-1)
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_

RF_PARAMS = {
    'model__n_estimators': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [None, 5, 10, 15, 20, 30],
    'model__min_samples_leaf': [1, 2, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__max_features': ['sqrt', 'log2', 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    'model__max_iter': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [3, 4, 5, 6, 8, None],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__min_samples_leaf': [5, 10, 20, 50, 100],
    'model__l2_regularization': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__max_leaf_nodes': [15, 20, 30, 40, 50, 60],
    'model__max_bins': [64, 128, 255],
}
XGB_PARAMS = {
    'model__n_estimators': [100, 200, 400, 600, 800],
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'model__min_child_weight': [1, 2, 5, 10],
    'model__gamma': [0, 0.1, 0.5, 1.0, 2.0],
    'model__reg_alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__reg_lambda': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 2.0],
}
MLP_PARAMS = {
    'model__hidden_layer_sizes': [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    'model__learning_rate_init': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    'model__learning_rate': ['constant', 'adaptive'],
    'model__batch_size': [32, 64, 128, 'auto'],
}

In [ ]:
for key, t in trainers.items():
    print(f'Tuning {key} ...')
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  'Random Forest (tuned)')
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  'Gradient Boosting (tuned)')
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, 'XGBoost (tuned)')
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, 'MLP (tuned)')

## Compare embeddings & pick best

In [ ]:
summary = []
for key, t in trainers.items():
    row = t.leaderboard().iloc[0]
    summary.append({'embedding': key, 'model': row['model'],
                    'validation_rmse': row['validation_rmse'], 'validation_mae': row['validation_mae']})
summary = pd.DataFrame(summary).sort_values('validation_rmse').reset_index(drop=True)
display(summary)
best = summary.iloc[0]
print('Best embedding+model:', best['embedding'], '|', best['model'], '| val_rmse', best['validation_rmse'])

## Save best model

In [ ]:
SAVE_DIR = PROJECT_ROOT / 'src' / 'models' / 'dataset_2' / '05_a'
path = trainers[best['embedding']].save_model(best['model'], SAVE_DIR)
print('Saved best (', best['embedding'], best['model'], ') ->', path)